# When care costs less, do people use more?

**A guided investigation · Chapter 1 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-rand_utilization.ipynb)

> **The question**  
> How does healthcare use differ across randomly assigned insurance plans?

An insurance plan changes the price a family faces when someone needs care. Lower prices may encourage useful treatment, but they can also increase spending without producing equal improvements in health. The RAND experiment assigned families to plans ranging from free care to substantial cost sharing. This extract joins healthcare-use records and annual spending, with expenses expressed in constant 1991 dollars. A row represents a participant-year, so the same person can contribute more than one record. Most people may spend relatively little while a small number incur very large expenses. That makes this a useful case for learning why a mean alone can be misleading. We will inspect the distribution, compare visits across plans, and ask what a spending difference does—and does not—tell us about the value of medical care.

### By the end, you will be able to…

- Explore skewed spending and compare visits and expenses across plans.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

Could two plans have similar typical spending but quite different average spending?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** Participant-year after inner joining claims and annual spending.  
**Stored size:** 20,203 rows × 11 columns.

| Column | What it records |
|---|---|
| `plan_type` | Assigned plan: 1 free care, 2 deductible, 3 coinsurance, 4 catastrophic coverage. Missing means unknown. |
| `visits` | Number of face-to-face medical visits in a participant-year. |
| `total_expenses` | Outpatient plus inpatient expenditure in constant 1991 dollars. |
| `admissions` | Number of hospital admissions in a participant-year. |

Rows are participant-years. Spending is in constant 1991 dollars. Repeated people and family assignments matter for inference; this lesson makes descriptive comparisons only.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch1/rand_utilization.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['plan_type', 'visits', 'total_expenses', 'admissions']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['plan_type', 'visits', 'total_expenses', 'admissions']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability_group': 'Ability group',
 'admissions': 'Admissions per participant-year',
 'census': 'Census year',
 'group': 'State group',
 'offer': 'Scholarship assignment',
 'plan': 'Assigned insurance plan',
 'state_name': 'Simulated state',
 'total_expenses': 'Annual expenses (1991 dollars)',
 'visits': 'Visits per participant-year'}

### Keep known plan assignments

We exclude unknown plan types from plan comparisons and translate codes into readable labels. `notna()` identifies nonmissing assignments; `map` replaces a code with a label. The earlier missing-value check still includes the full stored file.

In [ ]:
plans = df[df["plan_type"].notna()].copy()
plans["plan"] = plans["plan_type"].map({1: "Free care", 2: "Deductible", 3: "Coinsurance", 4: "Catastrophic"})

## Visual investigation 1 · Meet the long tail

**Predict:** Is most spending near the average, or is the distribution skewed?

A histogram counts observations inside value intervals. For spending, a few very large amounts can stretch the axis. Zoom to inspect the dense region, then double-click to restore the full distribution.

In [ ]:
fig = px.histogram(df, x="total_expenses", nbins=60, labels=labels, title="A small number of expensive participant-years")
fig.show()

**Explore the chart:** Drag across the lower-spending part of the chart to zoom. Reset and notice how far the tail extends.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

The long right tail means the mean can lie well above the spending of a typical participant-year. Do not delete large values merely to make the picture neater.

</details>

## Visual investigation 2 · Follow use, not only money

**Predict:** How do average medical visits compare across plans?

The mean is total visits divided by participant-years with a recorded value. It describes use, not the health benefit of those visits.

In [ ]:
visits = plans.groupby("plan", as_index=False)["visits"].mean()
fig = px.bar(visits, x="plan", y="visits", color="plan", labels=labels, title="Average visits by assigned plan")
fig.show()

**Explore the chart:** Hover to compare free care with each cost-sharing plan. State the unit: visits per participant-year.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Random assignment makes the plan contrast interesting, but these unadjusted summaries do not account for all analysis restrictions or repeated observations.

</details>

## Visual investigation 3 · Two summaries, two questions

**Predict:** Do mean and median spending tell the same story?

agg computes several summaries at once. melt reshapes those summaries into a long table: one row for each plan and summary type, which Plotly can color.

In [ ]:
spending = plans.groupby("plan")["total_expenses"].agg(["mean", "median"]).reset_index()
spending_long = spending.melt(id_vars="plan", var_name="summary", value_name="dollars")
fig = px.bar(spending_long, x="plan", y="dollars", color="summary", barmode="group", labels={"plan": "Assigned plan", "dollars": "Annual spending (1991 dollars)", "summary": "Summary"}, title="The mean and the typical participant-year")
fig.show()

**Explore the chart:** Toggle mean or median in the legend. Does your comparison depend on which summary you choose?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

The mean captures total resource use per row; the median locates the middle row. Neither tells us whether extra care was beneficial.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

In the visits comparison, replace visits with admissions and update the title.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Make a box plot comparing total expenses across plans, hiding individual points.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Use plans, plan, and total_expenses.

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

px.box(..., points=False) keeps the display compact.

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>fig = px.box(plans, x=&quot;plan&quot;, y=&quot;total_expenses&quot;, color=&quot;plan&quot;, points=False, labels=labels, title=&quot;Spending variation within plans&quot;)
fig.show()</code></pre>

The boxes show that a plan-level mean hides substantial variation. Keep the full data and explain any zoom used for display.

Copy the code into the starter cell to run it.

</details>

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**If free care increases visits, what additional outcomes would you need to decide whether it improves health?**

[Continue with Chapter 1: Randomized Trials](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/01-randomized-trials.html#step-2-estimate-causal-effects-on-health-care-use) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch1-rand-utilization-csv)

### Source and preparation notes

[Aron-Dine, Einav and Finkelstein (2013), The RAND Health Insurance Experiment, Three Decades Later](https://www.aeaweb.org/articles?id=10.1257/jep.27.1.197)

The preparation script inner-joins source files on person and year, maps 24 plans into four types, applies a year-specific CPI multiplier to expenses (constant 1991 dollars), sums total expense, and creates indicators/family ID. Profiles are unweighted.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.